# MNIST Digits Recognition - MLOps

## Kubeflow pipeline

In [ ]:
import os
import kfp
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, ClassificationMetrics
import numpy as np
from tensorflow import keras
import tensorflow as tf
import glob
from sklearn.metrics import confusion_matrix

from kubernetes import client 
from kserve import KServeClient
from kserve import constants
from kserve import utils
from kserve import V1beta1InferenceService
from kserve import V1beta1InferenceServiceSpec
from kserve import V1beta1PredictorSpec
from kserve import V1beta1TFServingSpec

from datetime import datetime
import time
import subprocess
import sys
from kale.common.serveutils import serve


In [4]:
# Specify proxy and no_proxy value if applicable
http_proxy="" ## "http://<proxy_host>:<port>"
# After default.svc append comma separated list of IP address, hostname, CIDR block, service name, etc.
no_proxy="" ## "localhost,.cluster.local,.svc,.default.svc, <host>:<port>, x.x.x.x/y,..."

#kale_shared_dir = "/marshal/" #skale internal data
user_shared_dir = "/mnt/data/" # volume user-pvc, also mounted by this jupyter notebook
training_data_subdir = "training/"
model_subdir = "model"
model_trained_subdir = "model_trained"
export_model_dir = "/mnt/export"

#Hyperparameters
num_epochs = 1
lr = 0.1

In [ ]:
import kfp
from pathlib import Path
kfp_client = kfp.Client()
user_mounted_dir_name = kfp_client.get_user_namespace()
user_shared_dir = f"{str(Path.home())}/{user_mounted_dir_name}/"
user_shared_dir

In [6]:
def pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [ ]:
x_train_artifact = final_training_data_dir + "xtrain.npy"
x_test_artifact = final_training_data_dir + "x_test.npy"
y_train_artifact = final_training_data_dir + "ytrain.npy"
y_test_artifact = final_training_data_dir + "y_test.npy"

if not os.path.exists(final_training_data_dir):
    os.makedirs(final_training_data_dir, 0o777)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    
np.save(x_train_artifact, x_train)
np.save(y_train_artifact, y_train)
np.save(x_test_artifact, x_test)
np.save(y_test_artifact, y_test)

In [11]:
final_training_data_dir = user_shared_dir + training_data_subdir

x_train_artifact = final_training_data_dir + "train-images-pickle.npy"
x_test_artifact = final_training_data_dir + "test-images-pickle.npy"
y_train_artifact = final_training_data_dir + "train-labels-pickle.npy"
y_test_artifact = final_training_data_dir + "test-labels-pickle.npy"


x_train_preproc = final_training_data_dir + "xtrain-preproc.npy"
x_test_preproc = final_training_data_dir + "xtest-preproc.npy"

# load data artifact store
x_train = np.load(x_train_artifact) 
x_test = np.load(x_test_artifact)
    
# reshaping the data
# reshaping pixels in a 28x28px image with greyscale, canal = 1. This is needed for the Keras API
x_train = x_train.reshape(-1,28,28,1)
x_test = x_test.reshape(-1,28,28,1)
# normalizing the data
# each pixel has a value between 0-255. Here we divide by 255, to get values from 0-1
x_train = x_train / 255
x_test = x_test / 255
    
#logging metrics using Kubeflow Artifacts
print("Len x_train", x_train.shape[0])
print("Len y_train", x_test.shape[0])
      
# save feuture in artifact store
np.save(x_train_preproc, x_train)
#os.rename("tmp/x_train.npy", x_train_processed.path)
    
np.save(x_test_preproc, x_test)
#os.rename("tmp/x_test.npy", x_test_processed.path)

Len x_train 60000
Len y_train 10000


In [ ]:
final_model_data_dir = user_shared_dir + model_subdir

if not os.path.exists(final_model_data_dir):
    os.makedirs(final_model_data_dir, 0o777)
    
model = keras.models.Sequential()
model.add(keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28,28,1)))
model.add(keras.layers.MaxPool2D(2, 2))

model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(64, activation='relu'))
model.add(keras.layers.Dense(32, activation='relu'))

model.add(keras.layers.Dense(10, activation='softmax'))
    
#saving model
model.save(final_model_data_dir)

In [ ]:
model_trained_dir = user_shared_dir + model_trained_subdir

if not os.path.exists(model_trained_dir):
    os.makedirs(model_trained_dir, 0o777)

#load dataset
x_train = np.load(x_train_preproc)
x_test = np.load(x_test_preproc)
y_train = np.load(y_train_artifact)
y_test = np.load(y_test_artifact)
    
#load model structure
model = keras.models.load_model(final_model_data_dir)
      
#compile the model - we want to have a binary outcome
model.compile(tf.keras.optimizers.SGD(learning_rate=lr),
              loss="sparse_categorical_crossentropy",
              metrics=['accuracy'])

    
#fit the model and return the history while training
history = model.fit(
      x = x_train,
      y = y_train,
      epochs = num_epochs,
      batch_size = 20,
)

     
# Test the model against the test dataset
# Returns the loss value & metrics values for the model in test mode.
model_loss, model_accuracy = model.evaluate(x=x_test,y=y_test)

print(f"model_loss={model_loss} model_accuracy={model_accuracy} ")


#build a confusione matrix
y_predict = model.predict(x=x_test)
y_predict = np.argmax(y_predict, axis=1)
print(f"y_predict={y_predict}")
    
#adding /1/ subfolder for TFServing
model_trained_dir = model_trained_dir + '/1/'
print("model_trained_uri: " + model_trained_dir)

export_model_dir = export_model_dir + '/1/'
print("model_export_uri: " + model_trained_dir)

# save model in local storage
keras.models.save_model(model, model_trained_dir, save_format='tf')

# save model in model export volume
keras.models.save_model(model, export_model_dir, save_format='tf')

# save model using TF API
#tf.saved_model.save(model, model_trained_dir)

In [12]:
#model_trained_dir = user_shared_dir + model_trained_subdir + '/1/'

In [ ]:
#Load model from the volume
model = keras.models.load_model(model_trained_dir)

test_accu = model.evaluate(x_test, y_test)
print('The testing accuracy is :',test_accu[1]*100, '%')

#preds = model.predict(x_test,verbose=1)
#print(preds)

#import time
#time.sleep(3600)

In [ ]:
model = tf.saved_model.load(model_trained_dir)
kfserver = serve(model)

In [ ]:
from kale.sdk import step
import subprocess

model_dir = "/mnt/export/"
model_volume = "model-volume-kale"  
model_name = "digitrec_model"

# Command to run the TensorFlow Serving container with the model
serving_command = f"""
    echo '
    apiVersion: v1
    kind: Pod
    metadata:
      name: tf-serve
    spec:
      containers:
      - name: tf-serving-container
        image: tensorflow/serving:latest
        ports:
        - containerPort: 8501
        args:
        - "--rest_api_port=8501"
        - "--model_name={model_name}"
        - "--model_base_path={model_dir}"
        volumeMounts:
        - name: {model_volume}
          mountPath: {model_dir}
      volumes:
      - name: {model_volume}
        persistentVolumeClaim:
          claimName: {model_volume}'  | kubectl create -f -
    """

# Run the serving command
result = subprocess.run(serving_command, shell=True, check=True)
    
# Check if the serving container is running successfully
if result.returncode == 0:
    print("TensorFlow Serving started successfully.")
else:
    raise Exception("Failed to start TensorFlow Serving.")

In [2]:
from kale.sdk import step
import subprocess

model_dir = "/mnt/export/"
model_volume = "model-volume-kale"  
model_name = "digitrec-serve"

api_version = 'serving.kserve.io/v1beta1'

serving_command = f"""
     echo '
      apiVersion: "{api_version}"
      kind: "InferenceService"
      metadata:
        name: {model_name}
        annotations:
          "sidecar.istio.io/inject": "false"
      spec:
        predictor:
          tensorflow:
            storageUri: "pvc://{model_volume}/"
            resources:
              limits:
                cpu: '2'
                memory: 1Gi
              requests:
                cpu: 100m
                memory: 50Mi'  | kubectl create -f -
      """

# Run the serving command
result = subprocess.run(serving_command, shell=True, check=True)
    
# Check if the serving container is running successfully
if result.returncode == 0:
    print("TensorFlow Serving started successfully.")
else:
    raise Exception("Failed to start TensorFlow Serving.")

inferenceservice.serving.kserve.io/digitrec-serve created
TensorFlow Serving started successfully.


### Next step: Inference test

Go the the following step: [Inference notebook](03-Inference.ipynb)